In [1]:
pip install datasets pandas

In [9]:
import gzip
import json
import urllib.request
import pandas as pd

meta_url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Pet_Supplies.jsonl.gz"

food_items = {}
req = urllib.request.urlopen(meta_url)

with gzip.GzipFile(fileobj=req) as f:
  for line in f:
    item = json.loads(line.decode("utf-8"))
    categories = " ".join(item.get("categories", [])).lower()

    if "food" in categories or "treats" in categories:
      food_items[item["parent_asin"]] = {
          "parent_asin": item["parent_asin"],
          "title": item.get("title"),
          "brand": item.get("store"),
          "categories": item.get("categories"),
          "price": item.get("price"),
          "features": item.get("features"),
          "description": item.get("description"),
      }

    if len(food_items) >= 15000:
      break

meta_df = pd.DataFrame(food_items.values())
meta_df.to_parquet("amazon_pet_food_meta.parquet", index=False)
print(f"Saved {len(meta_df)} pet food products.")

Saved 15000 pet food products.


In [10]:
review_url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Pet_Supplies.jsonl.gz"

target_asins = set(meta_df["parent_asin"])
matched_reviews = []
req_rev = urllib.request.urlopen(review_url)

with gzip.GzipFile(fileobj=req_rev) as f:
  for line in f:
    rev = json.loads(line.decode("utf-8"))
    if rev.get("parent_asin") in target_asins:
      matched_reviews.append({
          "parent_asin": rev["parent_asin"],
          "rating": rev.get("rating"),
          "title": rev.get("title"),
          "text": rev.get("text"),
          "verified_purchase": rev.get("verified_purchase"),
          "helpful_vote": rev.get("helpful_vote"),
      })

    if len(matched_reviews) >= 100000:
      break

reviews_df = pd.DataFrame(matched_reviews)
reviews_df.to_parquet("amazon_pet_food_reviews.parquet", index=False)
print(f"Saved {len(reviews_df)} matched reviews.")

Saved 100000 matched reviews.


In [13]:
import pandas as pd

# 1. Load your datasets (Fetching Open Pet Food Facts directly from the official URL)
opff_url = "https://static.openpetfoodfacts.org/data/en.openpetfoodfacts.org.products.csv.gz"
opff = pd.read_csv(
    opff_url, sep="\t", low_memory=False
)
amz_meta = pd.read_parquet("amazon_pet_food_meta.parquet")
amz_rev = pd.read_parquet("amazon_pet_food_reviews.parquet")

print(f"Open Pet Food Facts shape: {opff.shape}")
print(f"Amazon Meta shape: {amz_meta.shape}")
print(f"Amazon Reviews shape: {amz_rev.shape}")

# 2. Check key nutritional & ingredient columns in Open Pet Food Facts
key_cols = [
    c
    for c in opff.columns
    if any(
        k in c
        for k in [
            "product_name",
            "brands",
            "ingredients",
            "proteins",
            "fat",
            "fiber",
            "categories",
        ]
    )
]
print("\nKey OPFF Columns:", key_cols[:15])

Open Pet Food Facts shape: (15976, 204)
Amazon Meta shape: (15000, 7)
Amazon Reviews shape: (100000, 6)

Key OPFF Columns: ['product_name', 'abbreviated_product_name', 'brands_tags', 'brands_en', 'categories_tags', 'categories_en', 'ingredients_text', 'ingredients_tags', 'ingredients_analysis_tags', 'image_ingredients_url', 'image_ingredients_small_url', 'energy-from-fat_100g', 'fat_100g', 'saturated-fat_100g', 'unsaturated-fat_100g']


In [15]:
# Stage 1: Keep only the essential nutrition & ingredient columns
cols_to_keep = [
    "code",
    "product_name",
    "brands_en",
    "categories_en",
    "ingredients_text",
    "proteins_100g",
    "fat_100g",
    "fiber_100g",
]

# Filter to columns that exist in your dataframe and drop rows missing names or ingredients
existing_cols = [c for c in cols_to_keep if c in opff.columns]
clean_opff = (
    opff[existing_cols].dropna(subset=["product_name", "ingredients_text"]).copy()
)

# Clean up text formatting for easier searching later
clean_opff["ingredients_clean"] = clean_opff["ingredients_text"].str.lower()
clean_opff["categories_clean"] = (
    clean_opff["categories_en"].fillna("").str.lower()
)

print(f"Clean nutrition catalog ready: {len(clean_opff)} products")
clean_opff[[
    "product_name",
    "brands_en",
    "proteins_100g",
    "fat_100g",
    "ingredients_text",
]].head(5)

Clean nutrition catalog ready: 2708 products


,product_name,brands_en,proteins_100g,fat_100g,ingredients_text
7,Golden Age Flashcard,NaN,NaN,NaN,1234
30,gultair. dog. food,new,NaN,0.0,gultair. dog. food. only
43,Hill's r/d Weight Reduction,Hill's,31.1,10.6,"Viandes et sous-produits animaux, sous-pro..."
87,puppy toothpaste,Vet's Best,NaN,NaN,"purified water, glycerin, baking soda(sodium b..."
101,dog food,NaN,NaN,NaN,det hide treat your dog to the great taste and...


In [16]:
# Remove obvious non-food/test entries and short junk ingredient strings
non_food_keywords = (
    "toothpaste|flashcard|shampoo|litter|toy|collar|leash|brush|wipes"
)

clean_opff = clean_opff[
    (~clean_opff["product_name"].str.lower().str.contains(non_food_keywords, na=False))
    & (clean_opff["ingredients_clean"].str.len() > 15)
].copy()

print(f"Filtered food products: {len(clean_opff)}")
print(f"Products with Protein % data: {clean_opff['proteins_100g'].notna().sum()}")
print(f"Products with Fat % data: {clean_opff['fat_100g'].notna().sum()}")

Filtered food products: 2258
Products with Protein % data: 354
Products with Fat % data: 357


In [21]:
import re

# 1. Define allergen synonyms (including "poultry" under chicken)
ALLERGEN_SYNONYMS = {
    "chicken": ["chicken", "poultry", "poulet", "volaille", "volailles", "huhn", "hühner", "pollo"],
    "beef": ["beef", "boeuf", "bœuf", "bovin", "rind", "carne"],
    "salmon": ["salmon", "saumon", "lachs", "salmón"],
    "lamb": ["lamb", "agneau", "lamm", "cordero"],
}

# 2. Filter to clean English-only products
english_markers = r"\b(?:water|meat|meal|vitamins|salt|oil|fish|salmon|beef|lamb|rice|peas|potatoes)\b"
foreign_markers = r"\b(?:viandes|céréales|poissons|lachs|fleisch|sanglier|chèvre|sous-produits|zusammensetzung|blé|aliment|pour)\b"

clean_opff_en = clean_opff[
    clean_opff["ingredients_clean"].str.contains(english_markers, regex=True)
    & ~clean_opff["ingredients_clean"].str.contains(foreign_markers, regex=True)
    & ~clean_opff["product_name"].str.lower().str.contains(foreign_markers, regex=True)
].copy()

for protein, synonyms in ALLERGEN_SYNONYMS.items():
    pattern = r"\b(?:" + "|".join(synonyms) + r")\b"
    clean_opff_en[f"has_{protein}"] = clean_opff_en["ingredients_clean"].str.contains(pattern, regex=True)

print(f"Clean English-only catalog: {len(clean_opff_en)} products")

# 3. English-only Recommender Function
def recommend_foods_en(
    species="dog",
    avoid_ingredients=None,
    preferred_protein=None,
    max_fat_pct=None,
    top_n=5,
):
    df = clean_opff_en.copy()

    if species:
        df = df[
            df["categories_clean"].str.contains(species.lower(), na=False)
            | df["product_name"].str.lower().str.contains(species.lower(), na=False)
        ]

    if avoid_ingredients:
        for item in avoid_ingredients:
            terms = ALLERGEN_SYNONYMS.get(item.lower(), [item.lower()])
            pattern = r"\b(?:" + "|".join(map(re.escape, terms)) + r")\b"
            df = df[
                ~df["ingredients_clean"].str.contains(pattern, regex=True)
                & ~df["product_name"].str.lower().str.contains(pattern, regex=True)
            ]

    if max_fat_pct is not None:
        df = df[(df["fat_100g"] <= max_fat_pct) | (df["fat_100g"].isna())]

    df["match_score"] = 0
    if preferred_protein and f"has_{preferred_protein.lower()}" in df.columns:
        df["match_score"] += df[f"has_{preferred_protein.lower()}"].astype(int) * 3

    df["match_score"] += df["proteins_100g"].notna().astype(int)
    if max_fat_pct is not None:
        df["match_score"] += (df["fat_100g"] <= max_fat_pct).fillna(False).astype(int) * 2

    cols = ["product_name", "brands_en", "proteins_100g", "fat_100g", "match_score", "ingredients_text"]
    return df.sort_values(by=["match_score", "proteins_100g"], ascending=[False, False])[cols].head(top_n)

# Test it!
recommend_foods_en(
    species="dog",
    avoid_ingredients=["chicken"],
    preferred_protein="salmon",
    max_fat_pct=10.0,
    top_n=5,
)

Clean English-only catalog: 448 products


,product_name,brands_en,proteins_100g,fat_100g,match_score,ingredients_text
2381,Tyggeben,Chrisco,70.0,4.0,3,100 %beef rawhide
527,Wild Sea Catch,Earthborn,NaN,NaN,3,"Salmon, Pacific Whiting Meal, Whitefish Meal, ..."
673,Fancy Feast Kitten Tender Salmon Feast,Purina,NaN,NaN,3,"Salmon, liver, meat by-products, fish broth, f..."
1237,dog food,NaN,NaN,NaN,3,"salmon, fish meal, potatoes, lentils, peas, pe..."
1367,Raw diet for cats duck recipe,Northwest naturals,NaN,NaN,3,"Duck with Ground Bone, Duck Heart, Duck Liver,..."


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# 1. Clean review text and create weak-supervision labels for adverse reactions
rev_df = amz_rev.dropna(subset=["text", "rating"]).copy()
rev_df["full_text"] = (
    rev_df["title"].fillna("") + " " + rev_df["text"]
).str.lower()

# Flag reviews reporting sickness, digestive upset, or allergic reactions (with rating <= 3)
adverse_pattern = r"\b(?:vomit|vomiting|threw up|diarrhea|sick|ill|allergy|allergic|itch|itching|rash|hot spot|stomach|vet|recall|poison)\b"
rev_df["adverse_reaction"] = (
    rev_df["full_text"].str.contains(adverse_pattern, regex=True)
    & (rev_df["rating"] <= 3.0)
).astype(int)

print("Review breakdown (0 = Safe/Normal, 1 = Adverse Reaction Flagged):")
print(rev_df["adverse_reaction"].value_counts())

# 2. Split data and train a TF-IDF + Logistic Regression NLP classifier
X_train, X_test, y_train, y_test = train_test_split(
    rev_df["full_text"],
    rev_df["adverse_reaction"],
    test_size=0.2,
    random_state=42,
    stratify=rev_df["adverse_reaction"],
)

vectorizer = TfidfVectorizer(
    max_features=5000, ngram_range=(1, 2), stop_words="english"
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Use class_weight='balanced' because adverse reactions are a minority class
nlp_model = LogisticRegression(max_iter=1000, class_weight="balanced")
nlp_model.fit(X_train_vec, y_train)

# 3. Evaluate the classifier
y_pred = nlp_model.predict(X_test_vec)
print("\nNLP Adverse Reaction Classifier Performance:")
print(classification_report(y_test, y_pred))

Review breakdown (0 = Safe/Normal, 1 = Adverse Reaction Flagged):
adverse_reaction
0    97710
1     2290
Name: count, dtype: int64

NLP Adverse Reaction Classifier Performance:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     19542
           1       0.42      0.89      0.57       458

    accuracy                           0.97     20000
   macro avg       0.71      0.93      0.78     20000
weighted avg       0.98      0.97      0.97     20000



In [23]:
# 1. Predict adverse reaction probability across all 100,000 Amazon reviews
all_rev_vec = vectorizer.transform(rev_df["full_text"])
rev_df["risk_prob"] = nlp_model.predict_proba(all_rev_vec)[:, 1]

# Merge review risk scores with Amazon product metadata (to get brand & title)
amz_scored = rev_df.merge(
    amz_meta[["parent_asin", "title", "brand"]], on="parent_asin", how="left"
)
amz_scored["brand_clean"] = amz_scored["brand"].fillna("").str.lower().str.strip()

# Calculate brand-level customer safety stats (for brands with at least 20 reviews)
brand_safety = (
    amz_scored.groupby("brand_clean")
    .agg(
        total_reviews=("rating", "count"),
        avg_rating=("rating", "mean"),
        adverse_rate=("adverse_reaction", "mean"),
    )
    .reset_index()
)
brand_safety = brand_safety[brand_safety["total_reviews"] >= 20]
brand_safety["safety_score"] = (1 - brand_safety["adverse_rate"]) * 100


# 2. Final Unified Recommender: Nutrition + Allergen Filter + Amazon NLP Safety Score
def smart_pet_recommender(
    species="dog",
    avoid_ingredients=None,
    preferred_protein=None,
    max_fat_pct=None,
    top_n=5,
):
  # Get top nutrition & allergen matches from Open Pet Food Facts
  recs = recommend_foods_en(
      species=species,
      avoid_ingredients=avoid_ingredients,
      preferred_protein=preferred_protein,
      max_fat_pct=max_fat_pct,
      top_n=25,  # Pull top 25 candidates first, then enrich with safety data
  ).copy()

  # Match brand names to attach Amazon NLP customer safety scores
  recs["brand_clean"] = recs["brands_en"].fillna("").str.lower().str.strip()
  merged = recs.merge(brand_safety, on="brand_clean", how="left")

  # Fill missing brand stats with neutral baseline so unreviewed brands aren't dropped
  merged["safety_score"] = merged["safety_score"].fillna(97.5).round(1)
  merged["avg_rating"] = merged["avg_rating"].fillna(4.3).round(2)
  merged["total_reviews"] = merged["total_reviews"].fillna(0).astype(int)

  cols = [
      "product_name",
      "brands_en",
      "proteins_100g",
      "fat_100g",
      "match_score",
      "safety_score",
      "avg_rating",
      "total_reviews",
  ]
  return merged.sort_values(
      by=["match_score", "safety_score"], ascending=[False, False]
  )[cols].head(top_n)


# Test the complete end-to-end recommender!
smart_pet_recommender(
    species="dog",
    avoid_ingredients=["chicken"],
    preferred_protein="salmon",
    max_fat_pct=10.0,
    top_n=5,
)

,product_name,brands_en,proteins_100g,fat_100g,match_score,safety_score,avg_rating,total_reviews
8,Ocean Menu in Jelly,Whiskas,NaN,NaN,3,100.0,4.3,50
0,Tyggeben,Chrisco,70.0,4.0,3,97.5,4.3,0
1,Wild Sea Catch,Earthborn,NaN,NaN,3,97.5,4.3,0
3,dog food,NaN,NaN,NaN,3,97.5,4.3,0
5,Salmon & Potato Formula Food For Adult Dogs,4health,NaN,NaN,3,97.5,4.3,0


In [24]:
# Save processed tables for your Streamlit app & GitHub repo
clean_opff_en.to_parquet("clean_pet_nutrition.parquet", index=False)
brand_safety.to_parquet("brand_nlp_safety.parquet", index=False)
print(
    "Exported clean_pet_nutrition.parquet and brand_nlp_safety.parquet"
    " successfully!"
)

Exported clean_pet_nutrition.parquet and brand_nlp_safety.parquet successfully!
